In [5]:
con.close()

con = duckdb.connect("../taxi.duckdb")
con.execute("SELECT COUNT(*) FROM trips").fetchone()

(9554778,)

In [6]:
con.execute("""
    CREATE OR REPLACE TABLE zones AS
    SELECT * FROM read_csv_auto('../data/raw/taxi_zone_lookup.csv')
""")
con.execute("SELECT * FROM zones LIMIT 5").fetchdf()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [7]:
con.execute("""
    SELECT
      COUNT(*) AS total_rows,
      COUNT(*) FILTER (WHERE tpep_pickup_datetime < '2024-01-01'
                          OR tpep_pickup_datetime >= '2024-04-01') AS out_of_range,
      COUNT(*) FILTER (WHERE tpep_dropoff_datetime < tpep_pickup_datetime) AS dropoff_before_pickup,
      COUNT(*) FILTER (WHERE tpep_dropoff_datetime = tpep_pickup_datetime) AS zero_duration,
      COUNT(*) FILTER (WHERE tpep_dropoff_datetime >
                             tpep_pickup_datetime + INTERVAL 24 HOUR) AS over_24h,
      COUNT(*) FILTER (WHERE trip_distance <= 0) AS zero_or_neg_distance,
      COUNT(*) FILTER (WHERE fare_amount <= 0) AS zero_or_neg_fare,
      COUNT(*) FILTER (WHERE total_amount <= 0) AS zero_or_neg_total,
      COUNT(*) FILTER (WHERE passenger_count IS NULL) AS null_passengers,
      COUNT(*) FILTER (WHERE passenger_count = 0) AS zero_passengers,
      COUNT(*) FILTER (WHERE passenger_count > 6) AS over_6_passengers,
      COUNT(*) FILTER (WHERE PULocationID IS NULL) AS null_pickup_zone
    FROM trips
""").fetchdf()

,total_rows,out_of_range,dropoff_before_pickup,zero_duration,over_24h,zero_or_neg_distance,zero_or_neg_fare,zero_or_neg_total,null_passengers,zero_passengers,over_6_passengers,null_pickup_zone
0,9554778,21,180,2621,48,215764,139596,117187,751962,105931,99,0


In [8]:
# 1. Pickup zones not in the lookup table
con.execute("""
    SELECT COUNT(*) AS unmatched_pickup_zone
    FROM trips t
    LEFT JOIN zones z ON t.PULocationID = z.LocationID
    WHERE z.LocationID IS NULL
""").fetchdf()

,unmatched_pickup_zone
0,0


In [9]:
# 2. Which zone names are not real places (e.g. "Unknown", "N/A")?
con.execute("""
    SELECT z.LocationID, z.Borough, z.Zone, COUNT(*) AS trips
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    WHERE z.Borough = 'Unknown' OR z.Zone IN ('NV', 'NA', 'Unknown')
    GROUP BY ALL
    ORDER BY trips DESC
""").fetchdf()

,LocationID,Borough,Zone,trips
0,264,Unknown,N/A,31588


In [10]:
# 3. Do null passenger counts cluster by vendor?
con.execute("""
    SELECT VendorID,
           COUNT(*) AS trips,
           COUNT(*) FILTER (WHERE passenger_count IS NULL) AS null_passengers
    FROM trips
    GROUP BY VendorID
    ORDER BY VendorID
""").fetchdf()

,VendorID,trips,null_passengers
0,1,2334226,209226
1,2,7219832,542016
2,6,720,720


In [11]:
# 4. Exact duplicate rows (slow, give it a minute)
con.execute("""
    SELECT (SELECT COUNT(*) FROM trips) -
           (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM trips)) AS duplicate_rows
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_rows
0,1


In [12]:
con.execute("""
    SELECT tpep_pickup_datetime, tpep_dropoff_datetime, trip_distance,
           fare_amount, total_amount, PULocationID
    FROM trips
    WHERE trip_distance <= 0
    LIMIT 20
""").fetchdf()

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount,PULocationID
0,2024-01-01 00:52:09,2024-01-01 00:52:28,0.0,3.0,8.00,237
1,2024-01-01 00:14:29,2024-01-01 00:14:29,0.0,3.0,8.00,236
2,2024-01-01 00:58:50,2024-01-01 01:01:10,0.0,4.4,11.40,162
3,2024-01-01 00:15:16,2024-01-01 00:26:58,0.0,10.7,18.80,79
4,2024-01-01 00:39:34,2024-01-01 01:04:02,0.0,19.8,26.80,161
5,2024-01-01 00:18:26,2024-01-01 00:52:40,0.0,52.5,60.94,222
6,2024-01-01 00:04:05,2024-01-01 00:04:11,0.0,12.2,13.20,143
7,2024-01-01 00:25:58,2024-01-01 00:27:19,0.0,10.0,13.20,114
8,2024-01-01 00:31:10,2024-01-01 00:36:41,0.0,6.5,11.50,264
9,2024-01-01 00:28:27,2024-01-01 00:29:15,0.0,180.0,217.20,265


In [13]:
# 1. What are zones 264 and 265, and how many trips in each?
con.execute("""
    SELECT z.LocationID, z.Borough, z.Zone, COUNT(*) AS trips
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    WHERE z.LocationID IN (264, 265)
    GROUP BY ALL
""").fetchdf()

,LocationID,Borough,Zone,trips
0,265,N/A,Outside of NYC,5172
1,264,Unknown,N/A,31588


In [14]:
# 2. Split the zero-distance group by duration and fare
con.execute("""
    SELECT
      CASE WHEN date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime) < 60
           THEN 'under 1 min' ELSE '1 min or more' END AS duration_bucket,
      CASE WHEN fare_amount <= 0 THEN 'fare <= 0' ELSE 'fare > 0' END AS fare_bucket,
      COUNT(*) AS trips
    FROM trips
    WHERE trip_distance <= 0
    GROUP BY ALL
    ORDER BY trips DESC
""").fetchdf()

,duration_bucket,fare_bucket,trips
0,1 min or more,fare > 0,139215
1,under 1 min,fare > 0,64715
2,under 1 min,fare <= 0,10230
3,1 min or more,fare <= 0,1604


In [15]:
# 3. What payment types do the zero or negative fares have?
# (3 = no charge, 4 = dispute, per the TLC data dictionary)
con.execute("""
    SELECT payment_type, COUNT(*) AS trips
    FROM trips
    WHERE fare_amount <= 0
    GROUP BY 1
    ORDER BY trips DESC
""").fetchdf()

,payment_type,trips
0,4,70624
1,2,28247
2,0,21509
3,3,18697
4,1,519
